# Adaptive Boosting (AdaBoost)

**Notebook 1 of 4.** Bagging builds its models side by side and aggregates them. Boosting
builds them **one after another**, each one told where the ones before it went wrong.
AdaBoost does the telling with a **weight on every row**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

## 1. The data

`data/census.csv`: 45,222 records from the 1994 US census, five numeric columns and eight
text ones, with `income` saying whether the person earned more than $50,000 a year. A
quarter did, so answering `<=50K` to everything is already right **75.2%** of the time —
the number to beat.

A tree asks `feature <= threshold`, so the text columns are one-hot encoded first, turning
13 columns into 103.

In [ ]:
census = pd.read_csv("data/census.csv")

categorical = [
    "workclass", "education_level", "marital-status", "occupation",
    "relationship", "race", "sex", "native-country",
]

y = (census["income"] == ">50K").astype(int)

# astype(float) matters here: get_dummies returns booleans, and a mixed bool/int/float
# frame becomes an object array, which makes every fit below several times slower
X = pd.get_dummies(census.drop(columns=["income"]), columns=categorical).astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"{X.shape[1]} encoded columns, {len(X_train)} training rows")
print(f"answering <=50K to everything: {(y_test == 0).mean():.4f}")

## 2. A weak learner really is weak

Boosting is built out of **weak learners** — models only a little better than guessing. The
usual choice is a **decision stump**, a tree of depth 1, which asks one question and stops.

In [ ]:
# max_depth=1 is the whole tree: one question, two leaves, no second level
stump = DecisionTreeClassifier(max_depth=1, random_state=42)
stump.fit(X_train, y_train)

print(f"accuracy: {accuracy_score(y_test, stump.predict(X_test)):.4f}")
print(f"ROC AUC:  {roc_auc_score(y_test, stump.predict_proba(X_test)[:, 1]):.4f}")
print(f"labels it ever predicts: {np.unique(stump.predict(X_test))}")
print(f"the question it asks: {X.columns[stump.tree_.feature[0]]}")

**0.7521** — exactly the share earning `<=50K`, because the stump never predicts `>50K`.
It splits on being married: 45% of the married half are high earners against 7% of the
rest, and 45% is still a minority, so **both** leaves answer `<=50K`.

Its AUC is **0.7571** all the same, well clear of a coin toss. The stump **ranks** people
correctly and never commits, and that gap is what boosting exploits.

## 3. One round, by hand

AdaBoost keeps a weight $w_i$ on every training row $i$, starting equal at $1/n$ for $n$
rows. One round does four things:

1. fit a stump on the **weighted** rows;
2. measure its **weighted error** $\varepsilon = \sum_{i \text{ wrong}} w_i$, the weight
   sitting on the rows it got wrong;
3. give it a say $\alpha = \tfrac{1}{2}\ln\frac{1-\varepsilon}{\varepsilon}$, large when
   $\varepsilon$ is small;
4. multiply the weight of every wrong row by $e^{\alpha}$ and every right row by
   $e^{-\alpha}$, then rescale so the weights sum to 1.

Labels are written as $+1$ and $-1$ rather than 1 and 0, so that a weighted vote can be
read off by its sign.

In [ ]:
# +1 for >50K and -1 for <=50K, so the signs of a weighted vote mean something
s_train = np.where(y_train == 1, 1, -1)
s_test = np.where(y_test == 1, 1, -1)

X_tr, X_te = X_train.to_numpy(), X_test.to_numpy()

w = np.ones(len(X_tr)) / len(X_tr)       # every row equally important, to begin with

round_1 = DecisionTreeClassifier(max_depth=1, random_state=42)
round_1.fit(X_tr, s_train, sample_weight=w)

wrong = round_1.predict(X_tr) != s_train
epsilon = w[wrong].sum()                 # weighted error: the weight sitting on the misses
alpha = 0.5 * np.log((1 - epsilon) / epsilon)

print(f"{wrong.sum()} rows wrong out of {len(X_tr)}")
print(f"weighted error: {epsilon:.4f}")
print(f"say in the vote: {alpha:.4f}")

> **Exercise 1.** Apply the weight update. Multiply `w` by $e^{\alpha}$ where `wrong` is
> true and by $e^{-\alpha}$ where it is false, rescale so the result sums to 1, and store it
> as `w_next`. Print how far the weight of one wrong row and one right row moved.

<details><summary>
Click here for a hint...
</summary>

`np.where(wrong, 1.0, -1.0)` gives you the $+1/-1$ array for the exponent, so the update is
one `np.exp` call.

</details>

In [ ]:
# your code: build w_next from w, alpha and wrong, then report how the weights moved

<details><summary>
Click here to compare your answer...
</summary>

Wrong rows are multiplied by **2.02**, right ones by **0.66**, and the weight resting on
the rows the stump got wrong comes to **0.5000**.

That is the mechanism in one number, and it is no coincidence. The wrong rows carried
$\varepsilon$ and are scaled by $e^{\alpha} = \sqrt{(1-\varepsilon)/\varepsilon}$; the right
rows carried $1-\varepsilon$ and are scaled by $e^{-\alpha}$. Both products come to
$\sqrt{\varepsilon(1-\varepsilon)}$, so after rescaling each side holds exactly half.

The stump that has just run therefore scores **exactly 50%** on the reweighted data: it is
now a coin toss, and whatever the next stump finds has to be something this one missed.

</details>

## 4. One hundred rounds of it

The same four steps, in a loop. Each round's stump is kept with its $\alpha$, and a row is
predicted by the **sign of the weighted vote** $\sum_m \alpha_m h_m(x)$, writing $h_m$ for
the stump fitted in round $m$. The two practical points — clipping $\varepsilon$, and
accumulating the votes rather than recomputing them — are commented in the code.

In [ ]:
def adaboost(n_rounds):
    """Run AdaBoost on the census split, recording the error after every round.

    Args:
        n_rounds: how many stumps to fit, each one on the weights the last one left.

    Returns:
        DataFrame: one row per round, with the stump's weighted error and say, and
            the training and test error of the whole ensemble so far.
    """
    w = np.ones(len(X_tr)) / len(X_tr)
    score_tr = np.zeros(len(X_tr))       # the running weighted vote, one entry per row
    score_te = np.zeros(len(X_te))
    history = []

    for m in range(n_rounds):
        stump = DecisionTreeClassifier(max_depth=1, random_state=42)
        stump.fit(X_tr, s_train, sample_weight=w)

        wrong = stump.predict(X_tr) != s_train
        # clipped because a stump that is perfect, or perfectly wrong, sends alpha to infinity
        epsilon = np.clip(w[wrong].sum(), 1e-10, 1 - 1e-10)
        alpha = 0.5 * np.log((1 - epsilon) / epsilon)

        w = w * np.exp(alpha * np.where(wrong, 1.0, -1.0))
        w = w / w.sum()

        score_tr += alpha * stump.predict(X_tr)
        score_te += alpha * stump.predict(X_te)
        history.append({
            "round": m + 1,
            "epsilon": epsilon,
            "alpha": alpha,
            "train_error": (np.sign(score_tr) != s_train).mean(),
            "test_error": (np.sign(score_te) != s_test).mean(),
        })

    return pd.DataFrame(history)


history = adaboost(100)          # about 7 seconds
print(history.iloc[[0, 1, 2, 9, 24, 49, 99]].to_string(index=False, float_format="%.4f"))

Read the two middle columns down the page. The first stump scores $\varepsilon = 0.2478$
and earns a say of **0.5551**; by round 100 the error has drifted to **0.4975** and the say
to **0.0050**.

That drift is the algorithm working. Every round hands the next a harder version of the
problem, so late stumps are barely better than chance and are given almost no vote — and
the ensemble still improves, because faint opinions about *different* rows add up.

In [ ]:
plt.figure(figsize=(7, 4))
# both curves on one axis: the gap between them is what overfitting would widen
plt.plot(history["round"], history["train_error"], label="training")
plt.plot(history["round"], history["test_error"], label="test")
# the stump of section 2 got everything wrong that was >50K: 1 - 0.7521
plt.axhline(0.2479, color="grey", ls="dashed", label="one stump")
plt.xlabel("rounds")
plt.ylabel("error rate")
plt.legend()
plt.show()

Steep, then slow: the first 10 rounds take the test error from **0.2479** to **0.1620**,
and the next 90 buy **0.0105** more. The test curve tracks the training curve the whole way
instead of peeling away from it, which is the shape to want.

Final test error **0.1515**, so accuracy **0.8485**.

## 5. The library version

`AdaBoostClassifier` is the same algorithm, so it should land on the same number.

> **Note.** scikit-learn 1.9 renamed the weak learner argument to `estimator`, and dropped
> `algorithm=` along with the SAMME.R variant. Older tutorials pass `base_estimator=` and
> will raise a `TypeError`.

In [ ]:
ada = AdaBoostClassifier(
    # the same stump, the same 100 rounds, the same seed as the loop above
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    n_estimators=100,
    random_state=42,
)
ada.fit(X_train, y_train)

print(f"library:    {accuracy_score(y_test, ada.predict(X_test)):.4f}")
print(f"by hand:    {1 - history['test_error'].iloc[-1]:.4f}")
print(f"ROC AUC:    {roc_auc_score(y_test, ada.predict_proba(X_test)[:, 1]):.4f}")

**0.8485** both ways. The loop in section 4 is not a simplified illustration of
`AdaBoostClassifier`; it is what it does.

The AUC has gone from the single stump's **0.7571** to **0.9045**.

## 6. How weak should a weak learner be?

A stump is the traditional choice, but `estimator` takes any classifier and depth is the
dial: deeper trees are stronger alone, slower to fit, and readier to fit noise.

The comparison worth making is against **one tree, pruned properly**. If a hundred stumps
cannot beat that, the extra machinery has earned nothing.

In [ ]:
# ccp_alpha prunes the tree after it is grown; 1e-4 is the setting that scored best here
pruned = DecisionTreeClassifier(ccp_alpha=1e-4, random_state=42)
pruned.fit(X_train, y_train)

print(f"one pruned tree: {accuracy_score(y_test, pruned.predict(X_test)):.4f} accuracy, "
      f"{roc_auc_score(y_test, pruned.predict_proba(X_test)[:, 1]):.4f} AUC")

> **Exercise 2.** Fit `AdaBoostClassifier` with `n_estimators=150` at `max_depth` 1, 2 and
> 4, printing accuracy and ROC AUC for each. Which depths beat the pruned tree, and on
> which measure? It takes about a minute in all.

<details><summary>
Click here for a hint...
</summary>

Build a fresh `DecisionTreeClassifier(max_depth=d, random_state=42)` inside the loop.
`roc_auc_score` wants `predict_proba(X_test)[:, 1]`, the probability of the `>50K` class.

</details>

In [ ]:
# your code: AdaBoost at max_depth 1, 2 and 4, accuracy and AUC for each

<details><summary>
Click here to compare your answer...
</summary>

| weak learner | accuracy | ROC AUC |
| :-- | --: | --: |
| one pruned tree | 0.8537 | 0.9067 |
| 150 stumps, `max_depth=1` | 0.8493 | 0.9058 |
| 150 trees, `max_depth=2` | 0.8532 | 0.9090 |
| 150 trees, `max_depth=4` | **0.8608** | **0.9149** |

**Stumps lose.** A hundred and fifty of them reach 0.8493 against one tree's 0.8537, for
150 fits against one. Depth 2 draws level; depth 4 wins on both measures.

A weak learner can be too weak. A stump sees one column at a time and can never represent
*older **and** married*, so no weighted vote of stumps can either.

Depth, round count and learning rate are close to the whole of AdaBoost's control
surface.

</details>

## Summary

- AdaBoost fits weak learners **in sequence**, each on the **weights** left by the one before.
- A round turns the weighted error $\varepsilon$ into a say
  $\alpha = \tfrac{1}{2}\ln\frac{1-\varepsilon}{\varepsilon}$, then scales wrong rows by
  $e^{\alpha}$ and right rows by $e^{-\alpha}$.
- Both sides of that update come to $\sqrt{\varepsilon(1-\varepsilon)}$, so the stump that
  just ran scores **exactly 0.5000** and the next must find something else.
- Late stumps are near-useless alone, $\varepsilon = 0.4975$ and $\alpha = 0.0050$, and the
  ensemble still climbed from **0.7521** to **0.8485**.
- **A weak learner can be too weak.** Stumps lost to one pruned tree; depth 4 won.

Next, in `02_xgboost.ipynb`: gradient boosting, which fits the error itself rather than
reweighting rows, on a regression problem.